# 学习率调度与实践技巧

本notebook介绍学习率调度策略和优化算法的实践技巧。

## 学习目标

- 理解学习率调度的必要性
- 掌握常见的调度策略
- 学习批量大小的选择
- 了解优化器的实践技巧

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import matplotlib.pyplot as plt
import numpy as np
import math

torch.manual_seed(42)
np.random.seed(42)

## 1. 为什么需要学习率调度?

### 1.1 固定学习率的问题

**训练早期**:
- 远离最优解
- 梯度可能很大
- 需要大学习率快速前进

**训练后期**:
- 接近最优解
- 梯度较小
- 需要小学习率精细调整

**矛盾**: 同一个学习率无法同时满足两个阶段的需求!

### 1.2 学习率调度的目标

1. **早期**: 快速接近最优解
2. **中期**: 稳定下降
3. **后期**: 精细收敛

**类比**: 开车找停车位
- 高速公路: 快速行驶(大学习率)
- 停车场: 减速寻找(中学习率)
- 停车入位: 精细调整(小学习率)

In [ ]:
# 演示固定vs动态学习率
def f(x):
    """简单的凸函数"""
    return x**2

def f_grad(x):
    return 2 * x

def train_fixed_lr(lr, num_iters=50, x_init=10.0):
    """固定学习率"""
    x = x_init
    trajectory = [x]
    for _ in range(num_iters):
        x = x - lr * f_grad(x)
        trajectory.append(x)
    return trajectory

def train_scheduled_lr(lr_schedule, num_iters=50, x_init=10.0):
    """动态学习率"""
    x = x_init
    trajectory = [x]
    for t in range(num_iters):
        lr = lr_schedule(t)
        x = x - lr * f_grad(x)
        trajectory.append(x)
    return trajectory

# 定义调度策略
def exponential_decay(t, lr_init=0.5, decay_rate=0.95):
    return lr_init * (decay_rate ** t)

# 对比
traj_fixed = train_fixed_lr(lr=0.2, num_iters=50)
traj_scheduled = train_scheduled_lr(
    lambda t: exponential_decay(t, lr_init=0.5, decay_rate=0.95),
    num_iters=50
)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 参数轨迹
axes[0].plot([f(x) for x in traj_fixed], label='Fixed LR (0.2)', linewidth=2)
axes[0].plot([f(x) for x in traj_scheduled], label='Scheduled LR', linewidth=2)
axes[0].set_xlabel('Iteration')
axes[0].set_ylabel('f(x) = x²')
axes[0].set_title('目标函数值变化')
axes[0].legend()
axes[0].grid(True, alpha=0.3)
axes[0].set_yscale('log')

# 学习率变化
iterations = np.arange(50)
lrs = [exponential_decay(t) for t in iterations]
axes[1].plot(iterations, lrs, linewidth=2, color='green')
axes[1].axhline(0.2, color='red', linestyle='--', label='Fixed LR')
axes[1].set_xlabel('Iteration')
axes[1].set_ylabel('Learning Rate')
axes[1].set_title('学习率调度曲线')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("观察:")
print(f"固定学习率: 最终f(x) = {f(traj_fixed[-1]):.6f}")
print(f"调度学习率: 最终f(x) = {f(traj_scheduled[-1]):.10f}")
print("\n动态学习率可以更精确地收敛到最优解!")

## 2. 常见学习率调度策略

### 2.1 分段常数(Step Decay)

每隔固定epoch降低学习率:
$$
\eta_t = \eta_0 \times \gamma^{\lfloor t / T \rfloor}
$$

- $\eta_0$: 初始学习率
- $\gamma$: 衰减因子(如0.1)
- $T$: 衰减间隔(如30 epochs)

**优点**: 简单易用
**缺点**: 需要手动设置衰减点

### 2.2 指数衰减(Exponential Decay)

$$
\eta_t = \eta_0 \times \gamma^t
$$

- $\gamma \in (0, 1)$: 衰减率(如0.95, 0.99)

**优点**: 平滑衰减
**缺点**: 可能衰减过快

### 2.3 多项式衰减(Polynomial Decay)

$$
\eta_t = \eta_0 \times (1 + \alpha t)^{-\beta}
$$

常用: $\beta = 0.5$ (平方根衰减)
$$
\eta_t = \frac{\eta_0}{\sqrt{t+1}}
$$

**优点**: 理论保证(凸优化)
**缺点**: 后期衰减太慢

### 2.4 余弦退火(Cosine Annealing)

$$
\eta_t = \eta_{\min} + \frac{1}{2}(\eta_{\max} - \eta_{\min})\left(1 + \cos\left(\frac{t}{T}\pi\right)\right)
$$

**优点**: 平滑,自然
**应用**: 训练Transformer等大模型

### 2.5 预热(Warmup)

**问题**: 训练开始时参数随机,大学习率可能不稳定

**解决**: 从小学习率开始,线性增加到目标值
$$
\eta_t = \begin{cases}
\frac{t}{T_{\text{warmup}}} \eta_0 & t \leq T_{\text{warmup}} \\
\text{schedule}(t - T_{\text{warmup}}) & t > T_{\text{warmup}}
\end{cases}
$$

**典型**: Warmup 5-10% 总训练步数

In [ ]:
# 可视化各种学习率调度策略
num_epochs = 100
epochs = np.arange(num_epochs)

# 1. 分段常数
def step_decay(t, initial_lr=0.1, drop=0.5, epochs_drop=30):
    return initial_lr * (drop ** np.floor(t / epochs_drop))

# 2. 指数衰减
def exp_decay(t, initial_lr=0.1, decay_rate=0.95):
    return initial_lr * (decay_rate ** t)

# 3. 多项式衰减
def poly_decay(t, initial_lr=0.1, power=0.5):
    return initial_lr / ((t + 1) ** power)

# 4. 余弦退火
def cosine_annealing(t, initial_lr=0.1, min_lr=0.001, T_max=100):
    return min_lr + (initial_lr - min_lr) * 0.5 * (1 + np.cos(np.pi * t / T_max))

# 5. 带Warmup的余弦退火
def cosine_with_warmup(t, initial_lr=0.1, min_lr=0.001, warmup_epochs=10, T_max=100):
    if t < warmup_epochs:
        return (t / warmup_epochs) * initial_lr
    else:
        t_adjusted = t - warmup_epochs
        T_adjusted = T_max - warmup_epochs
        return min_lr + (initial_lr - min_lr) * 0.5 * (1 + np.cos(np.pi * t_adjusted / T_adjusted))

# 绘制
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
axes = axes.flatten()

schedules = [
    ('Step Decay', [step_decay(t) for t in epochs]),
    ('Exponential Decay', [exp_decay(t) for t in epochs]),
    ('Polynomial Decay (√t)', [poly_decay(t) for t in epochs]),
    ('Cosine Annealing', [cosine_annealing(t) for t in epochs]),
    ('Cosine + Warmup', [cosine_with_warmup(t) for t in epochs]),
]

for i, (name, lrs) in enumerate(schedules):
    axes[i].plot(epochs, lrs, linewidth=2.5)
    axes[i].set_xlabel('Epoch', fontsize=11)
    axes[i].set_ylabel('Learning Rate', fontsize=11)
    axes[i].set_title(name, fontsize=13, fontweight='bold')
    axes[i].grid(True, alpha=0.3)

# 第6个子图: 全部对比
for name, lrs in schedules:
    axes[5].plot(epochs, lrs, linewidth=2, label=name, alpha=0.8)

axes[5].set_xlabel('Epoch', fontsize=11)
axes[5].set_ylabel('Learning Rate', fontsize=11)
axes[5].set_title('All Schedules Comparison', fontsize=13, fontweight='bold')
axes[5].legend(fontsize=9)
axes[5].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("\n调度策略特点:")
print("1. Step Decay: 阶梯式下降,易于实现")
print("2. Exponential: 平滑下降,可能过快")
print("3. Polynomial: 理论保证,但后期慢")
print("4. Cosine: 平滑自然,大模型常用")
print("5. Cosine+Warmup: 稳定启动+平滑衰减,推荐!")

## 3. PyTorch中的学习率调度器

PyTorch在`torch.optim.lr_scheduler`模块提供了多种调度器:

### 3.1 常用调度器

In [ ]:
from torch.optim.lr_scheduler import (
    StepLR, ExponentialLR, CosineAnnealingLR,
    ReduceLROnPlateau, OneCycleLR
)

# 示例模型和优化器
model = nn.Linear(10, 1)
optimizer = optim.SGD(model.parameters(), lr=0.1)

# 1. StepLR: 每隔step_size个epoch衰减gamma倍
scheduler1 = StepLR(optimizer, step_size=30, gamma=0.1)

# 2. ExponentialLR: 每个epoch衰减gamma倍
scheduler2 = ExponentialLR(optimizer, gamma=0.95)

# 3. CosineAnnealingLR: 余弦退火
scheduler3 = CosineAnnealingLR(optimizer, T_max=100, eta_min=0.001)

# 4. ReduceLROnPlateau: 基于验证指标自适应调整
scheduler4 = ReduceLROnPlateau(optimizer, mode='min', factor=0.5,
                                patience=10, verbose=True)

# 5. OneCycleLR: 一个周期学习率(先增后减)
scheduler5 = OneCycleLR(optimizer, max_lr=0.1, steps_per_epoch=100, epochs=50)

print("PyTorch调度器使用方法:")
print("""
for epoch in range(num_epochs):
    train(...)  # 训练一个epoch
    
    # 对于大多数调度器
    scheduler.step()
    
    # 对于ReduceLROnPlateau
    # val_loss = validate(...)
    # scheduler.step(val_loss)
""")

### 3.2 实际训练示例

In [ ]:
# 生成训练数据
def generate_data(n_samples=1000):
    X = torch.randn(n_samples, 20)
    w_true = torch.randn(20, 1)
    y = X @ w_true + torch.randn(n_samples, 1) * 0.1
    return X, y

X_train, y_train = generate_data(1000)
X_val, y_val = generate_data(200)

# 训练函数
def train_with_scheduler(scheduler_type='StepLR', num_epochs=100):
    model = nn.Linear(20, 1)
    optimizer = optim.SGD(model.parameters(), lr=0.1, momentum=0.9)
    criterion = nn.MSELoss()
    
    # 选择调度器
    if scheduler_type == 'StepLR':
        scheduler = StepLR(optimizer, step_size=30, gamma=0.1)
    elif scheduler_type == 'ExponentialLR':
        scheduler = ExponentialLR(optimizer, gamma=0.95)
    elif scheduler_type == 'CosineAnnealingLR':
        scheduler = CosineAnnealingLR(optimizer, T_max=num_epochs, eta_min=0.001)
    elif scheduler_type == 'ReduceLROnPlateau':
        scheduler = ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=5)
    else:
        scheduler = None
    
    train_losses = []
    val_losses = []
    lrs = []
    
    for epoch in range(num_epochs):
        # 训练
        model.train()
        optimizer.zero_grad()
        outputs = model(X_train)
        loss = criterion(outputs, y_train)
        loss.backward()
        optimizer.step()
        train_losses.append(loss.item())
        
        # 验证
        model.eval()
        with torch.no_grad():
            val_outputs = model(X_val)
            val_loss = criterion(val_outputs, y_val)
            val_losses.append(val_loss.item())
        
        # 记录学习率
        lrs.append(optimizer.param_groups[0]['lr'])
        
        # 更新学习率
        if scheduler is not None:
            if scheduler_type == 'ReduceLROnPlateau':
                scheduler.step(val_loss)
            else:
                scheduler.step()
    
    return train_losses, val_losses, lrs

# 对比不同调度器
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

schedulers = ['None', 'StepLR', 'CosineAnnealingLR']
colors = ['red', 'blue', 'green']

for scheduler_name, color in zip(schedulers, colors):
    train_losses, val_losses, lrs = train_with_scheduler(scheduler_name, num_epochs=100)
    
    # 损失曲线
    axes[0].plot(val_losses, label=f'{scheduler_name}', linewidth=2, color=color)

axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Validation Loss')
axes[0].set_title('不同调度器的收敛曲线')
axes[0].legend()
axes[0].grid(True, alpha=0.3)
axes[0].set_yscale('log')

# 学习率曲线
for scheduler_name, color in zip(schedulers, colors):
    _, _, lrs = train_with_scheduler(scheduler_name, num_epochs=100)
    axes[1].plot(lrs, label=f'{scheduler_name}', linewidth=2, color=color)

axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Learning Rate')
axes[1].set_title('学习率变化')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("\n观察:")
print("- 固定学习率(None): 后期收敛慢")
print("- StepLR: 阶段性改善")
print("- CosineAnnealingLR: 平滑收敛,最终性能最好")

## 4. 批量大小(Batch Size)的选择

### 4.1 批量大小的影响

**小批量(如32, 64)**:
- ✅ 梯度噪声有助于探索
- ✅ 可能泛化更好
- ✅ 内存需求小
- ❌ 训练不稳定
- ❌ 每步时间长(GPU利用率低)

**大批量(如512, 1024, 2048)**:
- ✅ 梯度估计更准确
- ✅ 训练稳定
- ✅ GPU并行效率高
- ❌ 可能陷入尖锐最小值(泛化差)
- ❌ 内存需求大

### 4.2 线性缩放规则(Linear Scaling Rule)

**经验法则**: 批量大小增加k倍,学习率也增加k倍
$$
\eta_{\text{large batch}} = k \cdot \eta_{\text{small batch}}
$$

**原理**: 保持参数更新步长的期望不变

**例子**:
- Batch=32, LR=0.01
- Batch=256 (8倍), LR=0.08

**注意**: 需要配合Warmup!

In [ ]:
# 演示批量大小与学习率的关系
def train_with_batch_size(batch_size, lr, num_epochs=50):
    """使用指定批量大小和学习率训练"""
    X, y = generate_data(10000)
    dataset = torch.utils.data.TensorDataset(X, y)
    dataloader = torch.utils.data.DataLoader(dataset, batch_size=batch_size, shuffle=True)
    
    model = nn.Linear(20, 1)
    optimizer = optim.SGD(model.parameters(), lr=lr, momentum=0.9)
    criterion = nn.MSELoss()
    
    losses = []
    
    for epoch in range(num_epochs):
        epoch_loss = 0
        for batch_X, batch_y in dataloader:
            optimizer.zero_grad()
            outputs = model(batch_X)
            loss = criterion(outputs, batch_y)
            loss.backward()
            optimizer.step()
            epoch_loss += loss.item()
        
        losses.append(epoch_loss / len(dataloader))
    
    return losses

# 测试不同配置
configs = [
    (32, 0.01, 'Batch=32, LR=0.01'),
    (256, 0.01, 'Batch=256, LR=0.01 (不缩放)'),
    (256, 0.08, 'Batch=256, LR=0.08 (线性缩放)'),
]

plt.figure(figsize=(12, 6))

for batch_size, lr, label in configs:
    losses = train_with_batch_size(batch_size, lr, num_epochs=50)
    plt.plot(losses, label=label, linewidth=2)

plt.xlabel('Epoch')
plt.ylabel('Training Loss')
plt.title('批量大小与学习率的关系')
plt.legend()
plt.grid(True, alpha=0.3)
plt.yscale('log')
plt.show()

print("\n线性缩放规则:")
print("- 小批量(32) + 小学习率(0.01): 稳定但噪声大")
print("- 大批量(256) + 小学习率(0.01): 收敛太慢!")
print("- 大批量(256) + 缩放学习率(0.08): 快速且稳定!")

## 5. 优化实践技巧总结

### 5.1 优化器选择指南

| 场景 | 推荐优化器 | 配置 |
|------|------------|------|
| **通用/默认** | Adam | lr=1e-3, β₁=0.9, β₂=0.999 |
| **CV(图像)** | SGD+Momentum | lr=0.1, momentum=0.9, weight_decay=1e-4 |
| **NLP(Transformer)** | AdamW | lr=5e-5, warmup+cosine |
| **稀疏数据** | AdaGrad | lr=0.01 |
| **RNN** | RMSProp | lr=1e-3, α=0.9 |

### 5.2 学习率调参建议

**步骤**:
1. **找最大可用学习率**: 从小开始(1e-5),倍增测试,直到训练发散
2. **选择合适学习率**: 最大可用学习率的1/3到1/10
3. **配置调度器**: 
   - 计算机视觉: StepLR或MultiStepLR
   - NLP/Transformer: Warmup + Cosine
   - 一般任务: CosineAnnealingLR

### 5.3 批量大小建议

**经验法则**:
- 从32或64开始
- 根据GPU内存增大: 32 → 64 → 128 → 256
- 应用线性缩放规则调整学习率
- 大批量(>512)需要Warmup

### 5.4 训练技巧检查清单

**必须**:
- ✅ 使用学习率调度(至少StepLR)
- ✅ 监控训练和验证损失
- ✅ 使用适当的优化器(默认Adam)
- ✅ 设置合理批量大小

**推荐**:
- ✅ 梯度裁剪(防止梯度爆炸)
- ✅ 权重衰减(L2正则化)
- ✅ Early Stopping(防止过拟合)
- ✅ 学习率Warmup(大批量训练)

**高级**:
- ✅ 混合精度训练(加速)
- ✅ 梯度累积(模拟大批量)
- ✅ 学习率查找器(自动调参)

### 5.5 常见问题诊断

| 问题 | 可能原因 | 解决方案 |
|------|----------|----------|
| 训练loss不下降 | 学习率太小 | 增大学习率 |
| Loss震荡/发散 | 学习率太大 | 减小学习率,使用梯度裁剪 |
| 训练后期停滞 | 学习率太大 | 添加学习率衰减 |
| 训练慢,GPU利用率低 | 批量太小 | 增大批量,线性缩放学习率 |
| 训练快但泛化差 | 批量太大 | 减小批量,或使用更强正则化 |
| 梯度爆炸(NaN) | 学习率太大或网络不稳定 | 梯度裁剪,降低学习率,检查初始化 |

In [ ]:
# 完整的训练模板
class TrainingTemplate:
    """深度学习训练最佳实践模板"""
    
    def __init__(self, model, train_loader, val_loader, device='cuda'):
        self.model = model.to(device)
        self.train_loader = train_loader
        self.val_loader = val_loader
        self.device = device
        
        # 优化器(默认Adam)
        self.optimizer = optim.AdamW(
            model.parameters(),
            lr=1e-3,
            betas=(0.9, 0.999),
            weight_decay=1e-4  # L2正则化
        )
        
        # 学习率调度器(Cosine + Warmup)
        self.scheduler = optim.lr_scheduler.CosineAnnealingLR(
            self.optimizer,
            T_max=100,
            eta_min=1e-6
        )
        
        # 损失函数
        self.criterion = nn.CrossEntropyLoss()
        
        # Early stopping
        self.best_val_loss = float('inf')
        self.patience = 10
        self.patience_counter = 0
    
    def train_epoch(self):
        """训练一个epoch"""
        self.model.train()
        total_loss = 0
        
        for batch_idx, (data, target) in enumerate(self.train_loader):
            data, target = data.to(self.device), target.to(self.device)
            
            # 前向传播
            self.optimizer.zero_grad()
            output = self.model(data)
            loss = self.criterion(output, target)
            
            # 反向传播
            loss.backward()
            
            # 梯度裁剪(防止梯度爆炸)
            torch.nn.utils.clip_grad_norm_(self.model.parameters(), max_norm=1.0)
            
            # 参数更新
            self.optimizer.step()
            
            total_loss += loss.item()
        
        return total_loss / len(self.train_loader)
    
    def validate(self):
        """验证"""
        self.model.eval()
        total_loss = 0
        
        with torch.no_grad():
            for data, target in self.val_loader:
                data, target = data.to(self.device), target.to(self.device)
                output = self.model(data)
                loss = self.criterion(output, target)
                total_loss += loss.item()
        
        return total_loss / len(self.val_loader)
    
    def train(self, num_epochs=100):
        """完整训练循环"""
        train_losses = []
        val_losses = []
        
        for epoch in range(num_epochs):
            # 训练
            train_loss = self.train_epoch()
            train_losses.append(train_loss)
            
            # 验证
            val_loss = self.validate()
            val_losses.append(val_loss)
            
            # 学习率调度
            self.scheduler.step()
            current_lr = self.optimizer.param_groups[0]['lr']
            
            # Early stopping
            if val_loss < self.best_val_loss:
                self.best_val_loss = val_loss
                self.patience_counter = 0
                # 保存最佳模型
                torch.save(self.model.state_dict(), 'best_model.pth')
            else:
                self.patience_counter += 1
            
            if self.patience_counter >= self.patience:
                print(f"Early stopping at epoch {epoch}")
                break
            
            # 打印进度
            if (epoch + 1) % 10 == 0:
                print(f"Epoch {epoch+1}/{num_epochs}:")
                print(f"  Train Loss: {train_loss:.4f}")
                print(f"  Val Loss: {val_loss:.4f}")
                print(f"  LR: {current_lr:.6f}")
        
        return train_losses, val_losses

print("\n训练模板使用示例:")
print("""
# 创建模型和数据加载器
model = YourModel()
train_loader = ...
val_loader = ...

# 创建训练器
trainer = TrainingTemplate(model, train_loader, val_loader)

# 开始训练
train_losses, val_losses = trainer.train(num_epochs=100)
""")

## 6. 小结

### 核心要点

1. **学习率调度是必须的**
   - 早期大,后期小
   - 推荐: Cosine Annealing
   - 大批量训练需要Warmup

2. **优化器选择**
   - 默认: Adam/AdamW
   - CV任务: SGD+Momentum(可能更好)
   - Transformer: AdamW + Warmup + Cosine

3. **批量大小**
   - 从32/64开始
   - 增大时应用线性缩放规则
   - 大批量(>512)需要Warmup

4. **实践技巧**
   - 梯度裁剪
   - 权重衰减
   - Early Stopping
   - 监控训练曲线

### 推荐配置

**计算机视觉(ResNet, VGG等)**:
```python
optimizer = SGD(lr=0.1, momentum=0.9, weight_decay=1e-4)
scheduler = MultiStepLR(milestones=[30, 60, 90], gamma=0.1)
batch_size = 128
```

**NLP/Transformer**:
```python
optimizer = AdamW(lr=5e-5, betas=(0.9, 0.999), weight_decay=0.01)
scheduler = get_linear_schedule_with_warmup(warmup_steps=1000)
batch_size = 32
```

**通用任务**:
```python
optimizer = Adam(lr=1e-3)
scheduler = CosineAnnealingLR(T_max=100, eta_min=1e-6)
batch_size = 64
```

### 调试流程

1. 先用小数据集(100样本)过拟合 → 验证模型能力
2. 使用全数据,固定学习率训练 → 找合适初始学习率
3. 添加学习率调度 → 提升性能
4. 调整批量大小 → 优化训练速度
5. 添加正则化技巧 → 改善泛化

## 练习

1. **学习率查找器**: 实现自动寻找最优学习率的算法(从小到大,记录loss)。

2. **调度器对比**: 在MNIST上比较5种调度器(Step, Exp, Poly, Cosine, ReduceLROnPlateau)。

3. **线性缩放验证**: 验证批量大小从32增到256时,学习率从0.01增到0.08效果相同。

4. **Warmup实验**: 实现并测试不同Warmup长度(0%, 5%, 10%, 20%)的影响。

5. **完整训练**: 使用TrainingTemplate在CIFAR-10上训练ResNet,应用所有最佳实践。